In [1]:
import os
import base64
import json
import random 
import re
from openai import OpenAI
import anthropic
import pandas as pd
from tqdm import tqdm
import time
from dotenv import load_dotenv

# Load dataset

In [2]:
# Set paths relative to current working directory
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")
gif_paths = {}

def load_dataset(qa_json_path, description_csv_path):
    """Load and process Pororo dataset"""
    try:
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]
        descriptions = pd.read_csv(description_csv_path)
        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()

# TODO increase questions
def get_random_questions(qa_data, max_questions=40, base_pattern="Pororo_ENGLISH1", seed=42):
    random.seed(seed)

    # Filter all eligible questions
    filtered_questions = [q for q in qa_data if base_pattern in q["video_name"]]

    # Group by unique (video_name, supporting_num) pairs to avoid duplicates
    unique_pairs = {}
    for q in filtered_questions:
        key = (q["video_name"], q["supporting_num"])
        if key not in unique_pairs:
            unique_pairs[key] = []
        unique_pairs[key].append(q)

    # Sample from unique pairs
    unique_keys = list(unique_pairs.keys())
    selected_keys = random.sample(unique_keys, min(max_questions, len(unique_keys)))

    # Get one question from each selected pair
    sampled_questions = []
    episode_counts = {}

    for key in selected_keys:
        question = random.choice(unique_pairs[key])
        sampled_questions.append(question)
        episode = question["video_name"]
        episode_counts[episode] = episode_counts.get(episode, 0) + 1

    # Group by season for display
    season_episodes = {
        "Pororo_ENGLISH1_1": [],
        "Pororo_ENGLISH1_2": [],
        "Pororo_ENGLISH1_3": []
    }

    for ep in episode_counts.keys():
        season = "_".join(ep.split("_")[:3])
        if season in season_episodes:
            season_episodes[season].append(ep)

    # Print statistics
    print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
    for season in sorted(season_episodes.keys()):
        season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
        if season_eps:
            print(f"\n{season}:")
            for ep in sorted(season_eps.keys()):
                print(f"  {ep}: {season_eps[ep]} questions")

    return sampled_questions

def get_seeded_question(questions, gif_num, base_seed=42):
    """Get deterministic random question for a GIF"""
    if not questions:
        return None
    local_random = random.Random(base_seed + gif_num)
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

def encode_gif(gif_path):
    """Encode GIF file as base64 string"""
    try:
        if not os.path.exists(gif_path):
            print(f"Error: GIF not found at {gif_path}")
            return None
        with open(gif_path, "rb") as gif_file:
            image_base64 = base64.b64encode(gif_file.read()).decode('utf-8')
            return image_base64
    except Exception as e:
        print(f"Error encoding GIF: {e}")
        return None

# Load data
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Get random sample of questions TODO increase number of questions
sampled_questions = get_random_questions(qa_data, max_questions=40)

# Group questions by supporting_num
grouped_questions = {}
for entry in sampled_questions:
    video_name = entry["video_name"]
    supporting_num = entry["supporting_num"]
    key = (video_name, supporting_num)
    if key not in grouped_questions:
        grouped_questions[key] = []
    grouped_questions[key].append(entry)

# Get unique pairs to process
gif_pairs = sorted(list(grouped_questions.keys()))
correct_count = 0
total_count = len(gif_pairs)

# Used to store question information for each GIF pair
question_data = {}
for video_name, gif_num in gif_pairs:
    current_questions = grouped_questions[(video_name, gif_num)]
    if current_questions:
        entry = get_seeded_question(current_questions, int(gif_num))

        question = entry["question"]
        correct_idx = entry["correct_idx"]
        answers = [entry[f"answer{i}"] for i in range(5)]
        correct_answer = answers[correct_idx]
        qid = entry["qid"]

        question_data[(video_name, gif_num)] = {
            'entry': entry,
            'question': question,
            'correct_answer': correct_answer,
            'qid': qid
        }

        # gif path
        episode_parts = video_name.split("_")
        episode_folder = os.path.join(base_dir, "Scenes_Dialogues",
                                "_".join(episode_parts[:-1]),
                                video_name)
        gif_paths[(video_name, gif_num)] = os.path.join(episode_folder, f"{gif_num}.gif")

results_standard = []


Selected 40 questions from 22 episodes:

Pororo_ENGLISH1_1:
  Pororo_ENGLISH1_1_ep1: 1 questions
  Pororo_ENGLISH1_1_ep10: 2 questions
  Pororo_ENGLISH1_1_ep11: 1 questions
  Pororo_ENGLISH1_1_ep12: 4 questions
  Pororo_ENGLISH1_1_ep13: 2 questions
  Pororo_ENGLISH1_1_ep2: 4 questions
  Pororo_ENGLISH1_1_ep5: 1 questions
  Pororo_ENGLISH1_1_ep6: 4 questions
  Pororo_ENGLISH1_1_ep9: 1 questions

Pororo_ENGLISH1_2:
  Pororo_ENGLISH1_2_ep10: 1 questions
  Pororo_ENGLISH1_2_ep2: 2 questions
  Pororo_ENGLISH1_2_ep5: 2 questions
  Pororo_ENGLISH1_2_ep8: 3 questions

Pororo_ENGLISH1_3:
  Pororo_ENGLISH1_3_ep1: 1 questions
  Pororo_ENGLISH1_3_ep11: 1 questions
  Pororo_ENGLISH1_3_ep12: 2 questions
  Pororo_ENGLISH1_3_ep13: 1 questions
  Pororo_ENGLISH1_3_ep2: 1 questions
  Pororo_ENGLISH1_3_ep3: 1 questions
  Pororo_ENGLISH1_3_ep4: 1 questions
  Pororo_ENGLISH1_3_ep5: 2 questions
  Pororo_ENGLISH1_3_ep7: 2 questions


# Multi agent

In [3]:
load_dotenv()

# Configuration
# MODEL_NAME = "claude-3-5-haiku-20241022"
MODEL_NAME = "gpt-4o-mini"

is_openai_model = not MODEL_NAME.startswith("claude-")

if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Visual agent
def visual_agent(image_base64, max_retries=3, retry_delay=2):
    prompt = """
    As a cartoon visual expert, describe the image concisely and accurately.

    Guidelines: Analyze key cartoon elements including character design, facial expressions, compositional framing, color palette, and scene semantics.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url",
                            "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                visual_desc = completion.choices[0].message.content.strip()
                return visual_desc
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/gif",
                                "data": image_base64
                            }}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                visual_desc = completion.content[0].text.strip()
                return visual_desc

        except Exception as e:
            print(f"Visual agent attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    print("Error: Visual agent failed to process the image")
    return None

# Language agent
def language_agent(question, image_base64, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    prompt = f"""
    As a cartoon language expert, answer the question  using EXACTLY ONE SENTENCE:

    Input:
    Question: {question}
    Scene Description: {description}
    Visual Description: {visual_desc}
    Subtitles: {subtitles}

    Guidelines:
    1. No explanations allowed.
    2. Response must be in English only. DO NOT include text in any other language.
    3. Avoid phrases like "based on ...", "according to..." or "the description prided..."
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt},
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/gif;base64,{image_base64}"}
                                }
                            ],
                        }
                    ],
                    max_tokens=150,
                    temperature=0.1,
                )
                initial_predicted_answer = completion.choices[0].message.content.strip().lower()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image", "source": {"type": "base64", "media_type": "image/gif", "data": image_base64}},
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                initial_predicted_answer = completion.content[0].text.strip().lower()

            # Extract first sentence
            sentences = re.split(r'[.!?]', initial_predicted_answer)
            first_sentence = sentences[0].strip()

            # Skip empty sentences
            if not first_sentence and len(sentences) > 1:
                first_sentence = next((s.strip() for s in sentences if s.strip()), "")

            return first_sentence

        except Exception as e:
            print(f"Language agent attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                print("Error: Language agent failed to generate an answer")
            time.sleep(retry_delay)
            continue

    return None

# Hallucination detection agent
def hallucination_agent(question, image_base64, initial_predicted_answer, visual_desc, description, subtitles, max_retries=3, retry_delay=2):
    prompt = f"""
    As a cartoon hallucination detection expert, verify whether the predicted answer is accurate based on the available information.

    Input:
    Question: {question}
    Scene Description: {description}
    Dialogue/Subtitles: {subtitles}
    Image Context: {visual_desc}
    Predicted Answer: {initial_predicted_answer}

    Guidelines:
    1. Accuracy: Verify if the prediction is consistent with the image.
    2. Support: Check if the available evidence supports the prediction.
    3. Completeness: Ensure the answer contains the key information needed to answer the question.
    4. Error Analysis: If inaccuracies exist, identify what specific information was misunderstood or overlooked.
    5. Self-reflection: Consider why the model might have produced these inaccuracies (e.g., misleading visual cues, ambiguity).
    6. You MUST respond in ONE of these two formats ONLY:
       KEEP: [original answer] - if the prediction is accurate or you're uncertain
       REVISE: [one-sentence corrected answer] - ONLY if the prediction clearly contradicts evidence
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url",
                             "image_url": {"url": f"data:image/gif;base64,{image_base64}"}}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {
                                 "type": "base64",
                                 "media_type": "image/gif",
                                 "data": image_base64
                             }}
                        ]
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.content[0].text.strip()

            # Process the response
            # First attempt to match standard format
            keep_match = re.match(r'^KEEP:', response, re.IGNORECASE)
            revise_match = re.match(r'^REVISE:', response, re.IGNORECASE)

            if keep_match:
                final_answer = initial_predicted_answer.lower()
            elif revise_match:
                # Extract content following "REVISE:"
                content = response.split(":", 1)[1] if ":" in response else response.replace("REVISE", "", 1)
                content = content.strip().lower().rstrip('.!?')

                # Extract first sentence
                sentences = re.split(r'[.!?]', content)
                first_sentence = sentences[0].strip()

                # Skip empty sentences - use consistent approach with language_agent
                if not first_sentence and len(sentences) > 1:
                    first_sentence = next((s.strip() for s in sentences if s.strip()), "")

                final_answer = first_sentence
            else:
                # If format doesn't match, attempt a more flexible extraction approach
                if response.lower().startswith("keep"):
                    final_answer = initial_predicted_answer.lower()
                elif response.lower().startswith("revise"):
                    content = response.split(":", 1)[1] if ":" in response else response.replace("REVISE", "", 1)

                    # Extract first sentence
                    sentences = re.split(r'[.!?]', content)
                    first_sentence = sentences[0].strip()

                    # Skip empty sentences - use consistent approach with language_agent
                    if not first_sentence and len(sentences) > 1:
                        first_sentence = next((s.strip() for s in sentences if s.strip()), "")

                    final_answer = first_sentence.strip().lower().rstrip('.!?')
                else:
                    # Default to using the original answer
                    final_answer = initial_predicted_answer.lower()

            return final_answer

        except Exception as e:
            print(f"Hallucination check attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            continue

    # If all retry attempts fail, return the initial prediction
    return initial_predicted_answer.lower()

Using OpenAI model: gpt-4o-mini


# Compute accuracy

In [4]:
def compute_accuracy(question, correct_answer, predicted_answer, max_retries=2, retry_delay=2):
    prompt = f"""
    Evaluate the accuracy of the predicted answer according to criteria below:

    Input:
    Question: {question}
    Correct Answer: {correct_answer}
    Predicted Answer: {predicted_answer}

    Evaluation Rules:
    1. Focus PRIMARILY on semantic equivalence.
    2. Additional details should NEVER reduce the score if core information is correct.
    3. Judge based on whether the answer correctly addresses what the question asks for.
    4. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].

    Scoring Guide:
    - 1.0: Contains the correct core information, even if phrased differently or with additional details
    - 0.75: Mostly correct but missing minor information or containing slight inaccuracies
    - 0.5: Partially correct - contains some correct elements but misses important aspects
    - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
    - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering

    Scoring Examples:
    - Example of Score 1.0 (Perfect match or semantic equivalence):
    Question: "how did pororo feel after seeing that the flower has wilted"
    Correct: "he was very upset"
    Predicted: "pororo felt sad after seeing that the flower had wilted"
    Score: 1.0 (Synonyms with same core meaning)

    - Example of Score 1.0 (Additional details):
    Question: "what does crong do when pororo says 'come here'"
    Correct: "crong runs away from pororo"
    Predicted: "when pororo says 'come here,' crong tries to run away again"
    Score: 1.0 (Contains core information with additional details)

    - Example of Score 0.75 (Mostly correct but missing or slightly inaccurate information):
    Question: "what did loopy propose to the group after telling them about the flower"
    Correct: "loopy proposed that they should ask her anything"
    Predicted: "loopy proposed to the group that they ask the magic flower questions to predict the future"
    Score: 0.75 (Core action correct but adds slight inaccuracy about asking the flower directly)

    - Example of Score 0.5 (Partially correct):
    Question: "what does pororo almost forget to leave with poby"
    Correct: "the broken camera piece"
    Predicted: "pororo almost forgets to leave with poby's precious camera"
    Score: 0.5 (Mentions camera but misses the specific detail that it's broken)

    - Example of Score 0.25 (Slightly correct):
    Question: "what does eddy ask pororo"
    Correct: "he asks pororo what are you doing"
    Predicted: "eddy asks crong why pororo is acting so urgently"
    Score: 0.25 (Wrong recipient but related to pororo's actions)

    - Example of Score 0.0 (Completely incorrect):
    Question: "what was crong playing with as pororo entered the house"
    Correct: "crong was playing with a snowboard"
    Predicted: "crong was not shown playing with anything"
    Score: 0.0 (Directly contradicts the correct answer)
    """
    for attempt in range(max_retries):
        try:
            if is_openai_model:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.choices[0].message.content.strip()
            else:
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": prompt
                    }],
                    max_tokens=150,
                    temperature=0.1
                )
                response = completion.content[0].text.strip()

            numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
            if numeric_match:
                score = float(numeric_match.group(1))
            else:
                score = 0.0

            return score

        except Exception as e:
            print(f"Accuracy calculation attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    # Return 0 if it can't parse the score
    return 0.0

# Evaluate model performance

In [5]:
try:
    # Load dataset
    qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)
    # Initialize counters
    correct_count = 0
    total_count = len(gif_pairs)

    # Process each video and GIF pair
    for video_name, gif_num in tqdm(gif_pairs, total=total_count):
        # Get question information
        if (video_name, gif_num) not in question_data:
            print(f"No question data found for {video_name} GIF {gif_num}")
            continue

        # Use retrieved question information
        q_info = question_data[(video_name, gif_num)]
        question = q_info['question']
        correct_answer = q_info['correct_answer']
        qid = q_info['qid']

        # Get gif path
        gif_path = gif_paths[(video_name, gif_num)]
        gif_directory = os.path.dirname(gif_path)
        subtitles_path = os.path.join(gif_directory, "subtitles.txt")

        # Load subtitles
        with open(subtitles_path, "r") as f:
            subtitles = f.read()

        # Get description
        description_rows = descriptions.loc[
            (descriptions.iloc[:, 0] == video_name) &
            (descriptions.iloc[:, 1] == int(gif_num))
        ]
        if description_rows.empty:
            print(f"Description for {video_name} GIF {gif_num} not found")
            continue

        descriptions_list = description_rows.iloc[:, 2].tolist()
        description = " ".join(descriptions_list)

        # Encode GIF to base64
        image_base64 = encode_gif(gif_path)
        if image_base64 is None:
            print(f"Error: Failed to encode GIF for {video_name} GIF {gif_num}")
            continue

        # Multi-agent prediction process
        visual_desc = visual_agent(image_base64)
        if visual_desc is None:
            print(f"Error: Visual agent failed to process GIF {gif_num}")
            continue

        initial_predicted_answer = language_agent(question, image_base64, visual_desc, description, subtitles)
        if initial_predicted_answer is None:
            print(f"Error: Language agent failed to generate answer for {video_name} GIF {gif_num}")
            continue

        final_answer = hallucination_agent(
            question=question,
            image_base64=image_base64,
            initial_predicted_answer=initial_predicted_answer,
            visual_desc=visual_desc,
            description=description,
            subtitles=subtitles
        )

        predicted_answer = final_answer if final_answer else initial_predicted_answer

        # Calculate accuracy - ensure question parameter is passed
        is_correct = 0
        if predicted_answer is not None:
            is_correct = compute_accuracy(question, correct_answer, predicted_answer)
        correct_count += is_correct

        # Store current result
        result = {
            'gif_num': gif_num,
            'video_name': video_name,
            'qid': qid,
            'question': question,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'accuracy': is_correct
        }
        results_standard.append(result)

        # Print debugging info
        print(f"\nVideo name: {video_name}")
        print(f"GIF number: {gif_num}")
        print(f"QID: {qid}")
        print(f"Question: {question}")
        print(f"Correct Answer: {correct_answer}")
        print(f"Predicted Answer: {predicted_answer}")
        print(f"Accuracy: {float(is_correct):.4f}")

    # Calculate overall accuracy
    average_accuracy = correct_count / total_count if total_count > 0 else 0
    print(f"\nAverage Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error in evaluation: {e}")
    average_accuracy = 0

  2%|▎         | 1/40 [00:15<10:07, 15.58s/it]


Video name: Pororo_ENGLISH1_1_ep1
GIF number: 14
QID: 383
Question: whtat does eddy ask pororo
Correct Answer: he asks pororo what are you doing
Predicted Answer: eddy asks pororo the reason for his urgent action
Accuracy: 0.2500


  5%|▌         | 2/40 [00:24<07:29, 11.83s/it]


Video name: Pororo_ENGLISH1_1_ep10
GIF number: 12
QID: 1100
Question: were eddy's friends interested seeing his new toy?
Correct Answer: yes, they ran happily towards the new toy.
Predicted Answer: yes, eddy's friends were interested in seeing his new toy
Accuracy: 1.0000


  8%|▊         | 3/40 [00:34<06:37, 10.75s/it]

...


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 25
QID: 2333
Question: does the friends find pororo behind the snow man?
Correct Answer: the friends find pororo behind the snow man.
Predicted Answer: no, the friends do not find pororo behind the snowman
Accuracy: 0.0000


 98%|█████████▊| 39/40 [07:57<00:11, 11.34s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 25
QID: 2425
Question: what does crong do when pororo says "come here"
Correct Answer: crong runs away from pororo
Predicted Answer: crong tries to run away when pororo says "come here
Accuracy: 1.0000


100%|██████████| 40/40 [08:08<00:00, 12.21s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 47
QID: 2446
Question: what does poby say when invited to play
Correct Answer: he says "of course"
Predicted Answer: poby joyfully responds, "of course
Accuracy: 1.0000

Average Accuracy: 0.6438


# Save results

In [6]:
# Remove any existing Average rows
results_standard = [r for r in results_standard if r['gif_num'] != 'Average']

# Get unique videos and questions
unique_videos = len(set(r['video_name'] for r in results_standard))

# Add row numbers to each result
for i, result in enumerate(results_standard, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_standard) + 1,
    'gif_num': 'Average',
    'video_name': f'Total Videos: {unique_videos}',
    'qid': '',
    'question': f'Total Questions: {len(results_standard)}',
    'correct_answer': '',
    'predicted_answer': '',
    'accuracy': average_accuracy
}
results_standard.append(average_result)

# Define column order (reordered to put video_name before gif_num)
column_order = [
    'row_num',
    'video_name',
    'gif_num',
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'accuracy'
]

# Create safe model name for file naming
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Set up output directory
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
# Create standard subdirectory if it doesn't exist
standard_dir = os.path.join(results_dir, "standard")
os.makedirs(standard_dir, exist_ok=True)

output_path = os.path.join(
    results_dir,
    "standard",
    f'pororo_multi_agent_{safe_model_name}.csv'
)

# Remove existing file if it exists
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Save results with error handling
try:
    results_df = pd.DataFrame(results_standard)
    results_df = results_df[column_order]
    results_df.to_csv(output_path, index=False)

    print(f"Results successfully saved to: {output_path}")
    print(f"Average accuracy: {average_accuracy:.4f}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/pororo_multi_agent_gpt_4o_mini.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/pororo_multi_agent_gpt_4o_mini.csv
Average accuracy: 0.6438
